# Memecoins: Pump or Dump? Predict Pump.fun Launches on Solana

An offline CPU baseline for the community competition. This notebook contains
all runtime code: no repository, network access, external wallet data, serialized
models, or test outcomes are required. Attach the competition input and run all.

## Payoff and decision
Each token is skipped (`tp=0`) or assigned a take-profit in `[3,100]`. A reached
take returns `tp - 1.3`; otherwise it returns `max(final_x - 1.3, -1)`.
For `max_x=10, final_x=0.5`, choices `0,5,10,20` return `0,3.7,8.7,-0.8`.
Entry is the last AMM price at migration +60 seconds. Outcomes describe the
following hour, not the observed first-minute tape.

## Method and safeguards
Projected per-day parquet reads preserve chain/file order. Curve, first-minute
AMM, launch, migration and point-in-time creator/wallet/regime features feed
three-seed classifiers for reaches of 3/5/7/10/20x. Temporal held-out predictions
fit Platt calibration and held-payoff bins. A fixed take grid maximizes expected
value. Margins/exclusions are tuned on training OOF only; their reported PnL is
**in-sample-tuned**, with `default_pnl` (margin zero, no exclusions) and a tp=5
break-even baseline alongside it. Default PnL still uses OOF-fitted calibration
and held-payoff estimates; it is not nested unbiased validation.

Labels enter history only at `mt+3600`; each walk-forward fold trains strictly
before its day minus two hours. No test-wide ranking, quantiles, caps or outcome
lookup is used. Test decisions use training-fixed parameters independently.
Missing labels/failed modelling produce zero-take submissions; missing tapes
produce NaNs; LightGBM falls back to sklearn HGB.

**Warnings:** untested on real competition data. Synthetic numbers are NOT a
competition score. This is a community-run competition; prizes depend on the
individual sponsor and are not an official Kaggle payout. Check current rules,
including Kaggle foundational rule 6b about sharing competition code on Kaggle
forums/notebooks, before relying on a public GitHub repository for eligibility.
Not financial advice. See the project README for submission and freeze details.


In [ ]:
"""File-order tape features and point-in-time histories (all times UTC).

Bundled buy share uses a 1e15 raw-base-unit supply proxy. Missing inputs produce
NaNs, not fabricated zero activity. Histories use a fixed .14 smoothing prior,
never an outcome-derived global prior. Feature frames retain mint as their key.
"""
import os
from pathlib import Path
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

CURVE_COLUMNS = 'mint ts ts_rel direction sol tok wallet signing_wallet new_price failed slot tx_idx ix_idx'.split()
AMM_COLUMNS = 'mint ts_rel direction sol tok signing_wallet fee_payer new_price slot tx_idx pool_base_token_reserves_after pool_quote_token_reserves_after virtual_quote_reserves'.split()


def discover_data_dir(data_dir=None):
    """Return first candidate containing test.csv, or raise FileNotFoundError."""
    candidates = [data_dir, os.environ.get('PUMP_DATA_DIR'),
                  '/kaggle/input/competitions/memecoins-pump-or-dump',
                  '/kaggle/input/memecoins-pump-or-dump']
    candidates += sorted(Path('/kaggle/input').glob('*')) + sorted(Path('/kaggle/input').glob('*/*'))
    candidates += ['data', 'data/synthetic', '../data']
    for candidate in candidates:
        if candidate is not None and (Path(candidate)/'test.csv').is_file():
            return Path(candidate)
    raise FileNotFoundError('No data directory containing test.csv')


def discover_tapes(data_dir):
    """Map UTC day -> {curve: [paths], amm: [paths]} for both layouts."""
    result = {}
    for path in sorted(Path(data_dir).rglob('*.parquet')):
        if re.match(r'0.*_sample_', path.name):
            continue
        match = re.search(r'(?:day_|day=)(\d{4}-\d{2}-\d{2})', str(path))
        if match:
            kind = 'amm' if ('pumpswap' in str(path).lower() or 'amm' in str(path).lower()) else 'curve'
            result.setdefault(match[1], {'curve': [], 'amm': []})[kind].append(path)
    return result


def _csv(root, name):
    path = Path(root)/name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def _ratio(a, b):
    return a / b if pd.notna(b) and b != 0 else np.nan


def _series(df, key):
    return df[key] if key in df else pd.Series(np.nan, index=df.index)


def _aggregate(frame, kind):
    """Vectorized per-day reductions; first/last always retain input order."""
    original = set(frame)
    g = frame.copy()
    index = pd.Index(g.mint.unique(), name='mint')
    f = pd.DataFrame(index=index)
    for col in set(CURVE_COLUMNS + AMM_COLUMNS) - set(g):
        g[col] = np.nan
    if kind == 'curve':
        bad = g[g.failed.eq(1)]
        gb = bad.groupby('mint', sort=False)
        f['n_failed'] = gb.size().reindex(index, fill_value=0)
        f['failed_frac'] = g.failed.ne(0).groupby(g.mint, sort=False).mean()
        f['failed_buy_share'] = bad.direction.eq('buy').groupby(bad.mint, sort=False).mean()
        f['unique_failed_wallets'] = gb.wallet.nunique().reindex(index, fill_value=0)
        g = g[g.failed.eq(0)].copy()
    else:
        g = g[g.ts_rel.between(0, 60)].copy()
    g['w'] = g.wallet if kind == 'curve' else g.signing_wallet
    g['buy'] = g.direction.isin(['buy', 'B'])
    g['sell'] = g.direction.isin(['sell', 'S'])
    g['buy_sol'] = g.sol.where(g.buy, 0.)
    g['sell_sol'] = g.sol.where(g.sell, 0.)
    groups = g.groupby('mint', sort=False)
    total = groups.sol.sum()
    buyvol = groups.buy_sol.sum()
    f['n_trades'] = groups.size().reindex(index, fill_value=0)
    f['volume_sol'] = total
    f['buy_share'] = buyvol / total.replace(0, np.nan)
    f['n_wallets'] = groups.w.nunique()
    wb = g[g.buy].groupby(['mint', 'w'], sort=False).sol.sum()
    shares = wb / wb.groupby(level=0, sort=False).transform('sum').replace(0, np.nan)
    f['top1_share'] = shares.groupby(level=0, sort=False).max()
    f['hhi'] = shares.pow(2).groupby(level=0, sort=False).sum()
    pairs = g.groupby(['mint', 'w'], sort=False)[['buy', 'sell']].max()
    f['roundtrip_wallet_frac'] = (pairs.buy & pairs.sell).groupby(level=0, sort=False).sum() / f.n_wallets.replace(0, np.nan)
    prices = g[g.new_price.gt(0)].groupby('mint', sort=False).new_price.agg(['first', 'last', 'max', 'min'])
    f['price_last'] = prices['last']
    if kind == 'curve':
        f['n_buys'] = groups.buy.sum()
        f['n_sells'] = groups.sell.sum()
        f['trades_per_wallet'] = f.n_trades / f.n_wallets.replace(0, np.nan)
        ordered = shares.sort_values(ascending=False, kind='stable')
        top = ordered.groupby(level=0, sort=False).cumcount().lt(3)
        f['top3_share'] = ordered[top].groupby(level=0, sort=False).sum()
        f['largest_buy_sol'] = g[g.buy].groupby('mint', sort=False).sol.max()
        f['largest_buy_share'] = f.largest_buy_sol / buyvol.replace(0, np.nan)
        f['curve_age_s'] = -groups.ts_rel.first()
        for name, mask in [('last10min_share', g.ts_rel.gt(-600)),
                           ('last60s_pre_share', g.ts_rel.ge(-60) & g.ts_rel.lt(0))]:
            f[name] = g.sol.where(mask, 0.).groupby(g.mint, sort=False).sum() / total.replace(0, np.nan)
        f['sell_volume_share'] = groups.sell_sol.sum() / total.replace(0, np.nan)
        f['n_sellers_frac'] = g[g.sell].groupby('mint', sort=False).w.nunique() / f.n_wallets.replace(0, np.nan)
    else:
        for name, lo, hi in [('w1', 0, 15), ('w2', 15, 30), ('w3', 30, 60)]:
            mask = g.ts_rel.ge(lo) & (g.ts_rel.le(hi) if name == 'w3' else g.ts_rel.lt(hi))
            volume = g.sol.where(mask, 0.).groupby(g.mint, sort=False).sum()
            f['volume_'+name] = volume
            f['buy_share_'+name] = g.buy_sol.where(mask, 0.).groupby(g.mint, sort=False).sum() / volume.replace(0, np.nan)
        f['momentum'] = np.log1p(f.volume_w3 / (f.volume_w1 + 1e-12))
        f['price_first'] = prices['first']
        for name in ['last', 'max', 'min']:
            f['price_'+name+'_rel'] = prices[name] / prices['first']
        f['drawdown'] = prices['last'] / prices['max'] - 1
        g['depth'] = g.pool_quote_token_reserves_after + g.virtual_quote_reserves
        depth = g.groupby('mint', sort=False).depth.agg(['first', 'last'])
        f['depth_first'], f['depth_last'] = depth['first'], depth['last']
        f['depth_growth'] = depth['last'] / depth['first']
        clustered = g.groupby(['mint', 'slot'], sort=False).sol.transform('size').ge(2)
        f['slot_clustering_share'] = g.sol.where(clustered, 0.).groupby(g.mint, sort=False).sum() / total.replace(0, np.nan)
        f['sponsored_share'] = g.fee_payer.ne(g.signing_wallet).groupby(g.mint, sort=False).mean()
        f['net_buy_sol'] = buyvol - groups.sell_sol.sum()
    if 'direction' not in original:
        for col in f:
            if any(x in col for x in ['buy', 'sell', 'top', 'hhi', 'roundtrip']):
                f[col] = np.nan
    if 'sol' not in original:
        for col in f:
            if any(x in col for x in ['volume', 'buy_share', 'top', 'hhi', 'momentum', 'largest', 'clustering', 'last10', 'last60', 'net_buy']):
                f[col] = np.nan
    if 'fee_payer' not in original and 'sponsored_share' in f:
        f['sponsored_share'] = np.nan
    if kind == 'curve' and 'failed' not in original:
        f.loc[:, :] = np.nan
    return f.add_prefix(kind+'_')


def build_features(tokens, data_dir):
    """Return (one-row-per-token feature frame, mint/wallet early-buyers table).

    Only partitions needed by tokens are read, one day at a time. No label
    columns are copied. Parquet projection intersects each actual schema.
    """
    tokens = tokens[['mint', 'mt']].drop_duplicates('mint').reset_index(drop=True)
    result = tokens[['mint']].copy()
    result['hour'] = pd.to_datetime(tokens.mt, unit='s', utc=True).dt.hour
    records, early = {}, []
    tapes = discover_tapes(data_dir)
    days = pd.to_datetime(tokens.mt, unit='s', utc=True).dt.strftime('%Y-%m-%d')
    for day in days.unique():
        needed = set(tokens.loc[days.eq(day), 'mint'])
        for kind, columns in [('curve', CURVE_COLUMNS), ('amm', AMM_COLUMNS)]:
            parts = []
            for path in tapes.get(day, {}).get(kind, []):
                available = pq.read_schema(path).names
                cols = [c for c in columns if c in available]
                if 'mint' not in cols:
                    continue
                part = pd.read_parquet(path, columns=cols)
                parts.append(part[part.mint.isin(needed)])
            if not parts:
                continue
            frame = pd.concat(parts, ignore_index=True)
            if kind == 'curve' and 'wallet' not in frame and 'signing_wallet' in frame:
                frame['wallet'] = frame.signing_wallet
            records.setdefault(kind, []).append(_aggregate(frame, kind))
            if kind == 'curve' and {'failed', 'direction', 'wallet'} <= set(frame):
                buyers = frame.loc[frame.failed.eq(0) & frame.direction.eq('buy'), ['mint', 'wallet']].dropna().drop_duplicates()
                early.append(buyers[buyers.groupby('mint', sort=False).cumcount().lt(20)])
    template = pd.concat([_aggregate(pd.DataFrame(columns=CURVE_COLUMNS), 'curve'),
                          _aggregate(pd.DataFrame(columns=AMM_COLUMNS), 'amm')], axis=1)
    numeric = template.reindex(tokens.mint).astype(float)
    for kind, parts in records.items():
        values = pd.concat(parts)
        numeric.loc[:, values.columns] = values.reindex(tokens.mint).to_numpy()
    result = pd.concat([result, numeric.reset_index(drop=True)], axis=1)
    result['price_jump'] = result.amm_price_last / result.curve_price_last
    creation = _csv(data_dir, 'pumpfun_token_creation.csv')
    meta = tokens.merge(creation, on='mint', how='left') if 'mint' in creation else tokens.copy()
    result['launch_log_age'] = np.log1p((meta.mt - _series(meta, 'created_ts')).clip(lower=0))
    for c in ['bundle_size', 'bundled_buys', 'bundled_buys_count', 'direct_pf_invocation']:
        result[c] = _series(meta, c)
    result['log_dev_balance'] = np.log1p(_series(meta, 'dev_balance') / 1e9)
    result['bundled_buys_share'] = _series(meta, 'bundled_buys') / 1e15
    migrations = _csv(data_dir, 'pfamm_migrations.csv')
    if 'mint' not in migrations and 'bonding_curve' in migrations and {'mint', 'curve_address'} <= set(creation):
        migrations = migrations.merge(creation[['mint', 'curve_address']], left_on='bonding_curve', right_on='curve_address', how='left')
    mig = tokens.merge(migrations, on='mint', how='left') if 'mint' in migrations else tokens.copy()
    result['migration_log_sol'] = np.log1p(_series(mig, 'sol_amount') / 1e9)
    return result, pd.concat(early, ignore_index=True) if early else pd.DataFrame(columns=['mint', 'wallet'])


def _prefix(events, queries, value_columns, delay=3600, strict=False):
    """Sorted per-key prefix counts/sums; no label aggregation over the future."""
    output = np.zeros((len(queries), 1+len(value_columns)))
    e = events.dropna(subset=['key']).sort_values(['key', 'mt'], kind='stable').copy()
    if e.empty or queries.empty:
        return output
    e['count'] = e.groupby('key', sort=False).cumcount()+1
    for col in value_columns:
        e[col] = e.groupby('key', sort=False)[col].cumsum()
    e['event_time'] = e.mt + delay
    left = queries[['key', 'mt']].copy()
    left['position'] = np.arange(len(left))
    left = left.dropna(subset=['key']).sort_values('mt', kind='stable')
    if left.empty:
        return output
    merged = pd.merge_asof(left, e[['key', 'event_time', 'count']+value_columns].sort_values('event_time', kind='stable'),
                           left_on='mt', right_on='event_time', by='key', direction='backward',
                           allow_exact_matches=not strict)
    output[merged.position.to_numpy()] = merged[['count']+value_columns].fillna(0).to_numpy()
    return output


def history_features(query_df, labeled_df, universe_df=None, early_buyers=None,
                     creation_df=None, prior=.14, smoothing=5.):
    """Point-in-time creator, wallet and trailing-day regime features.

    universe_df includes all known mint/mt pairs; creator may be supplied there
    or via creation_df. Labels resolve at mt+3600 (inclusive). Prior universe
    launches use strictly mt<query.mt. Inputs are never modified.
    """
    q = query_df[['mint', 'mt']].reset_index(drop=True)
    universe = (universe_df if universe_df is not None else query_df).copy()
    if creation_df is not None and {'mint', 'creator'} <= set(creation_df):
        creators = creation_df[['mint', 'creator']].drop_duplicates('mint')
    elif 'creator' in universe:
        creators = universe[['mint', 'creator']].drop_duplicates('mint')
    else:
        creators = pd.DataFrame(columns=['mint', 'creator'])
    labels = labeled_df.reindex(columns=['mint', 'mt', 'max_x', 'final_x']).copy()
    if labels.empty:
        labels = pd.DataFrame(columns=['mint', 'mt', 'max_x', 'final_x'])
    labels = labels.drop_duplicates('mint')
    labels['hit'] = (labels.max_x >= 3).astype(float)
    labels['hit5'] = (labels.max_x >= 5).astype(float)
    labels['logmax'] = np.log(labels.max_x.clip(lower=1e-12).astype(float))
    labels['logfinal'] = np.log(labels.final_x.clip(lower=1e-12).astype(float))
    qc = q.merge(creators, on='mint', how='left').rename(columns={'creator': 'key'})
    lc = labels.merge(creators, on='mint', how='left').rename(columns={'creator': 'key'})
    c = _prefix(lc, qc, ['hit', 'logmax'])
    uc = universe[['mint', 'mt']].drop_duplicates('mint').merge(creators, on='mint', how='left').rename(columns={'creator': 'key'})
    launches = _prefix(uc, qc, [], delay=0, strict=True)[:, 0]
    out = q[['mint']].copy()
    out['creator_resolved_count'] = c[:, 0]
    out['creator_hit_rate'] = (c[:, 1]+prior*smoothing)/(c[:, 0]+smoothing)
    out['creator_mean_log_max'] = np.divide(c[:, 2], c[:, 0], out=np.full(len(q), np.nan), where=c[:, 0]>0)
    out['creator_prior_launches'] = launches
    missing = qc.key.isna()
    out.loc[missing, out.columns[1:]] = np.nan
    for col in ['wallet_hit_rate', 'wallet_prior_appearances', 'wallet_with_history_frac']:
        out[col] = np.nan
    if early_buyers is not None and not early_buyers.empty:
        early = early_buyers[['mint', 'wallet']].drop_duplicates()
        eq = q.reset_index(names='query_index').merge(early, on='mint').rename(columns={'wallet': 'key'})
        el = labels.merge(early, on='mint').rename(columns={'wallet': 'key'})
        wc = _prefix(el, eq, ['hit'])
        eq['rate'] = (wc[:, 1]+prior*smoothing)/(wc[:, 0]+smoothing)
        eq['count'] = wc[:, 0]
        eq['seen'] = wc[:, 0]>0
        summary = eq.groupby('query_index', sort=False).agg(wallet_hit_rate=('rate', 'mean'), wallet_prior_appearances=('count', 'sum'), wallet_with_history_frac=('seen', 'mean'))
        out.loc[summary.index, summary.columns] = summary.to_numpy()
    labels = labels.sort_values('mt', kind='stable')
    times = labels.mt.to_numpy(dtype=float)
    cumulative = {c: np.r_[0., labels[c].cumsum().to_numpy()] for c in ['hit', 'hit5']}
    lo = np.searchsorted(times, q.mt.to_numpy()-3600-86400, side='left')
    hi = np.searchsorted(times, q.mt.to_numpy()-3600, side='right')
    n = hi-lo
    out['regime_count'] = n
    for col, source in [('regime_hit3', 'hit'), ('regime_hit5', 'hit5')]:
        out[col] = np.divide(cumulative[source][hi]-cumulative[source][lo], n,
                             out=np.full(len(q), np.nan), where=n>0)
    # Rolling median on a merged event/query timeline handles ties inclusively.
    events = labels[['mt', 'logfinal']].rename(columns={'mt': 'time', 'logfinal': 'value'}).copy()
    events['position'] = -1
    events['order'] = 0
    queries = pd.DataFrame({'time': q.mt.to_numpy()-3600, 'value': np.nan,
                            'position': np.arange(len(q)), 'order': 1})
    timeline = pd.concat([events, queries], ignore_index=True).sort_values(['time', 'order'], kind='stable')
    if len(timeline):
        series = pd.Series(timeline.value.to_numpy(), index=pd.to_datetime(timeline.time, unit='s'))
        med = series.rolling('86400s', closed='both').median().to_numpy()
        selected = timeline.position.ge(0).to_numpy()
        values = np.full(len(q), np.nan)
        values[timeline.loc[selected, 'position'].to_numpy(dtype=int)] = med[selected]
        out['regime_median_log_final'] = values
    else:
        out['regime_median_log_final'] = np.nan
    return out


In [ ]:
"""Competition payoff and fixed, row-independent expected-value policy."""
import numpy as np
import pandas as pd

LEVELS = np.array([3., 5., 7., 10., 20.])
TP_GRID = np.array([3., 4., 5., 6., 7., 8., 10., 12., 15., 20., 30., 50., 100.])


def pnl(tp, max_x, final_x):
    """Exact Evaluation payoff, scalar or broadcasting numpy arrays."""
    tp, max_x, final_x = np.broadcast_arrays(tp, max_x, final_x)
    result = np.where(tp == 0, 0., np.where(max_x >= tp, tp-1.3, np.maximum(final_x-1.3, -1.)))
    return float(result) if result.ndim == 0 else result


def monotone_probs(probabilities):
    return np.minimum.accumulate(np.clip(np.asarray(probabilities, dtype=float), 1e-8, 1-1e-8), axis=-1)


def reach_probability(probabilities, tp, alpha=1.5):
    """Log-log interpolation between levels; power-law extrapolation above 20."""
    p = monotone_probs(probabilities)
    t = float(tp)
    if t > LEVELS[-1]:
        return np.clip(p[..., -1]*(20/t)**alpha, 1e-8, 1-1e-8)
    j = int(np.clip(np.searchsorted(LEVELS, t, side='right')-1, 0, len(LEVELS)-2))
    weight = np.clip(np.log(t/LEVELS[j])/np.log(LEVELS[j+1]/LEVELS[j]), 0, 1)
    return np.exp((1-weight)*np.log(p[..., j])+weight*np.log(p[..., j+1]))


class HeldPayoff:
    """Conditional failed-take payoff table with training-fixed P3 bins."""

    def fit(self, labels, probabilities):
        p3 = np.asarray(probabilities)[:, 0]
        self.edges = np.unique(np.quantile(p3, np.linspace(0, 1, 6)[1:-1]))
        bins = np.searchsorted(self.edges, p3, side='right')
        held = np.maximum(labels.final_x.to_numpy()-1.3, -1.)
        fallback = float(held.mean()) if len(held) else -1.
        self.values = np.full((len(self.edges)+1, len(TP_GRID)), fallback)
        for j, tp in enumerate(TP_GRID):
            missed = labels.max_x.to_numpy() < tp
            unconditional = held[missed].mean() if missed.any() else fallback
            for b in range(len(self.edges)+1):
                mask = missed & (bins == b)
                self.values[b, j] = held[mask].mean() if mask.any() else unconditional
        return self

    def predict(self, probabilities):
        bins = np.searchsorted(self.edges, np.asarray(probabilities)[..., 0], side='right')
        return self.values[bins]


def expected_values(probabilities, held, alpha=1.5):
    p = np.stack([reach_probability(probabilities, t, alpha) for t in TP_GRID], axis=-1)
    return p*(TP_GRID-1.3)+(1-p)*held


class DecisionPolicy:
    """OOF-tuned thresholds/margin; inference never uses batch statistics."""

    def __init__(self, alpha=1.5):
        self.alpha = alpha
        self.margin = 0.
        self.rules = []

    @staticmethod
    def _excluded(features, rules):
        mask = np.zeros(len(features), dtype=bool)
        for col, threshold, high in rules:
            if col in features:
                v = features[col].to_numpy()
                mask |= v > threshold if high else v < threshold
        return mask

    def fit(self, features, probabilities, labels):
        self.held = HeldPayoff().fit(labels, probabilities)
        ev = expected_values(probabilities, self.held.predict(probabilities), self.alpha)
        best = ev.argmax(axis=1)
        scores = ev[np.arange(len(ev)), best]
        chosen = TP_GRID[best]
        gains = pnl(chosen, labels.max_x.to_numpy(), labels.final_x.to_numpy())
        candidates = []
        candidate_columns = [('bundled_buys_share', True), ('bundled_buys', True),
                             ('curve_curve_age_s', False), ('amm_drawdown', False), ('amm_top1_share', True)]
        self.available_candidates = [col for col, _ in candidate_columns if col in features and features[col].notna().any()]
        print(f'Available exclusion candidates: {self.available_candidates}')
        for col, high in candidate_columns:
            if col in features and features[col].notna().any():
                for quantile in ([.9, .95] if high else [.05, .1]):
                    candidates.append((col, float(features[col].quantile(quantile)), high))
        best_pnl = -np.inf
        for margin in [0., .1, .25, .5, 1., 2.]:
            rules = []
            active = scores > margin
            score = gains[active].sum()
            while True:
                options = [(gains[active & ~self._excluded(features, [r])].sum(), r) for r in candidates if r not in rules]
                if not options:
                    break
                value, rule = max(options, key=lambda pair: pair[0])
                if value <= score+1e-12:
                    break
                rules.append(rule)
                active &= ~self._excluded(features, [rule])
                score = value
            if score > best_pnl:
                best_pnl, self.margin, self.rules = score, margin, rules
        return self

    def predict(self, features, probabilities):
        ev = expected_values(probabilities, self.held.predict(probabilities), self.alpha)
        j = ev.argmax(axis=1)
        active = (ev[np.arange(len(ev)), j] > self.margin) & ~self._excluded(features, self.rules)
        return np.where(active, TP_GRID[j], 0.).astype(float)

    def decide_row(self, features, probabilities):
        """Pure single-token decision given a frozen fitted policy."""
        return float(self.predict(pd.DataFrame([features]), np.asarray(probabilities).reshape(1, -1))[0])


def validate_submission(submission, test):
    """Raise ValueError for any contract violation; otherwise return True."""
    if list(submission.columns) != ['mint', 'tp']:
        raise ValueError('submission columns must be mint,tp')
    if submission.mint.duplicated().any() or submission.isna().any().any():
        raise ValueError('NaNs or duplicate mints')
    if submission.mint.tolist() != test.mint.tolist():
        raise ValueError('mint order/set differs from test')
    values = pd.to_numeric(submission.tp, errors='coerce').to_numpy()
    if not np.isfinite(values).all() or not ((values == 0) | ((values >= 3) & (values <= 100))).all():
        raise ValueError('tp outside {0} U [3,100]')
    return True


In [ ]:
"""Temporal level models and an offline, graceful-fallback pipeline."""
from pathlib import Path
from dataclasses import dataclass
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import contextlib


def _thread_limit(n=4):
    """Limit BLAS/OpenMP threads when sklearn exposes a controller; otherwise no-op."""
    try:
        from sklearn.utils.parallel import _get_threadpool_controller
        return _get_threadpool_controller().limit(limits=n)
    except Exception:
        return contextlib.nullcontext()






def _matrix(features):
    return features.drop(columns=['mint'], errors='ignore').replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)


class LevelModels:
    """Three-seed bagging with sparse-positive scaled lower-level fallback."""

    def __init__(self, use_lightgbm=True):
        self.use_lightgbm = use_lightgbm

    def fit(self, features, labels):
        self.models = []
        x = _matrix(features)
        self.rates = np.array([(labels.max_x >= level).mean() for level in LEVELS])
        self.backend = 'sklearn HGB'
        classifier = None
        if self.use_lightgbm:
            try:
                from lightgbm import LGBMClassifier
                classifier = LGBMClassifier
                self.backend = 'LightGBM'
            except ImportError:
                pass
        with _thread_limit(4):
            for level in LEVELS:
                y = (labels.max_x.to_numpy() >= level).astype(int)
                fitted = []
                if y.sum() >= 15 and (1-y).sum() >= 15:
                    for seed in [0, 1, 2]:
                        if classifier is not None:
                            estimator = classifier(num_leaves=15, learning_rate=.05, n_estimators=300,
                                min_child_samples=40, subsample=.8, bagging_freq=1,
                                colsample_bytree=.8, reg_lambda=1., n_jobs=4, verbosity=-1, random_state=seed)
                        else:
                            estimator = HistGradientBoostingClassifier(max_leaf_nodes=15, learning_rate=.05,
                                max_iter=300, l2_regularization=1., random_state=seed, early_stopping=False)
                        estimator.fit(x, y)
                        fitted.append(estimator)
                self.models.append(fitted)
        r10, r20 = self.rates[-2:]
        self.alpha = float(np.clip(np.log(r10/r20)/np.log(2), .2, 5)) if r20 > 0 and r10 > r20 else 1.5
        return self

    def predict(self, features):
        x = _matrix(features)
        result = np.empty((len(x), len(LEVELS)))
        with _thread_limit(4):
            for j, fitted in enumerate(self.models):
                if fitted:
                    result[:, j] = np.mean([m.predict_proba(x)[:, 1] for m in fitted], axis=0)
                elif j == 0:
                    result[:, j] = self.rates[j]
                else:
                    result[:, j] = result[:, j-1] * self.rates[j]/max(self.rates[j-1], 1e-12)
        return monotone_probs(result)


class PlattCalibrator:
    """Per-level logistic calibration fit only on temporal held-out predictions."""

    @staticmethod
    def _logit(p):
        p = np.clip(p, 1e-6, 1-1e-6)
        return np.log(p/(1-p)).reshape(-1, 1)

    def fit(self, probabilities, labels):
        self.models = []
        for j, level in enumerate(LEVELS):
            y = (labels.max_x.to_numpy() >= level).astype(int)
            model = None
            if len(np.unique(y)) == 2:
                model = LogisticRegression(C=1., solver='lbfgs').fit(self._logit(probabilities[:, j]), y)
            self.models.append(model)
        return self

    def predict(self, probabilities):
        result = np.asarray(probabilities).copy()
        for j, model in enumerate(self.models):
            if model is not None:
                result[:, j] = model.predict_proba(self._logit(result[:, j]))[:, 1]
        return monotone_probs(result)


def _read(root, name, columns):
    path = root/name
    return pd.read_csv(path) if path.exists() else pd.DataFrame(columns=columns)


def _write_submission(submission, out_paths):
    if isinstance(out_paths, (str, Path)):
        out_paths = [out_paths]
    for path in out_paths:
        try:
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            submission.to_csv(path, index=False)
        except OSError as error:
            print(f'Could not write {path}: {error}')


@dataclass
class FittedArtifacts:
    """In-memory frozen fit and original history context; never serialized."""

    model: LevelModels
    calibration: PlattCalibrator
    policy: DecisionPolicy
    feature_columns: list
    training_labels: pd.DataFrame
    universe: pd.DataFrame
    creation: pd.DataFrame
    early_buyers: pd.DataFrame
    base_features: pd.DataFrame
    walk_forward: pd.DataFrame

    def predict_features(self, features, individually=False):
        ordered = features.reindex(columns=self.feature_columns)
        probabilities = self.calibration.predict(self.model.predict(ordered))
        if individually:
            return np.array([self.policy.decide_row(row, probability)
                             for row, probability in zip(ordered.to_dict('records'), probabilities)])
        return self.policy.predict(ordered, probabilities)


def load_training_labels(data_dir, archive_days=30):
    """Read the same eligible train/archive labels used by the original pipeline."""
    root = Path(data_dir)
    train = _read(root, 'train.csv', ['mint', 'mt', 'max_x', 'final_x'])
    archive = _read(root, 'train_archive.csv', ['mint', 'mt', 'max_x', 'final_x'])
    if archive_days > 0 and not archive.empty and not train.empty:
        dates = pd.to_datetime(archive.mt, unit='s', utc=True).dt.strftime('%Y-%m-%d')
        tape_days = {day for day, parts in discover_tapes(root).items() if parts['curve'] or parts['amm']}
        archive = archive[(archive.mt >= train.mt.min()-archive_days*86400) & dates.isin(tape_days)]
        train = pd.concat([archive, train], ignore_index=True)
    train = train.drop_duplicates('mint').dropna(subset=['mt', 'max_x', 'final_x']).sort_values('mt').reset_index(drop=True)
    return train


def fit_pipeline(data_dir, wf_days=7, archive_days=30, use_lightgbm=True,
                 training_labels=None, test_tokens=None):
    """Fit once, returning artifacts; errors propagate for evaluation callers."""
    root = Path(data_dir)
    train = load_training_labels(root, archive_days) if training_labels is None else training_labels
    test = _read(root, 'test.csv', ['mint', 'mt']) if test_tokens is None else test_tokens
    if len(train) < 200 or test.empty or test.mt.isna().any():
        raise ValueError('Insufficient labels or missing test times for a frozen fit')
    universe = pd.concat([train[['mint', 'mt']], test[['mint', 'mt']]], ignore_index=True).drop_duplicates('mint')
    creation = _read(root, 'pumpfun_token_creation.csv', ['mint', 'creator'])
    start = time.perf_counter()
    base, early = build_features(universe, root)
    base = base.set_index('mint')

    def features_for(rows, visible):
        history = history_features(rows, visible, universe, early, creation).set_index('mint')
        return base.reindex(rows.mint).reset_index().merge(history.reset_index(), on='mint', how='left')

    training_features = features_for(train, train)
    print(f'Features: {len(universe)} tokens, {len(training_features.columns)-1} columns, {time.perf_counter()-start:.3f}s')
    days = np.sort((train.mt.to_numpy()//86400).astype(int))
    fold_days = np.unique(days)[-max(0, int(wf_days)):] if wf_days > 0 else []
    predictions, indices, day_names, oof_parts = [], [], [], []
    for day in fold_days:
        fit_mask = train.mt < day*86400-7200
        val_mask = (train.mt >= day*86400) & (train.mt < (day+1)*86400)
        if fit_mask.sum() < 200 or not val_mask.any():
            continue
        fit_labels = train[fit_mask]
        model = LevelModels(use_lightgbm).fit(training_features.loc[fit_mask], fit_labels)
        valid = train[val_mask]
        valid_features = features_for(valid, fit_labels)
        predictions.append(model.predict(valid_features))
        oof_parts.append(valid_features)
        indices.extend(valid.index.tolist())
        day_names.extend([pd.to_datetime(day*86400, unit='s', utc=True).strftime('%Y-%m-%d')]*len(valid))
    if not predictions:
        raise ValueError('No eligible temporal OOF folds; cannot tune safely')
    oof_labels = train.loc[indices].reset_index(drop=True)
    oof_features = pd.concat(oof_parts, ignore_index=True)
    raw = np.concatenate(predictions)
    calibration = PlattCalibrator().fit(raw, oof_labels)
    calibrated = calibration.predict(raw)
    final_model = LevelModels(use_lightgbm).fit(training_features, train)
    policy = DecisionPolicy(final_model.alpha).fit(oof_features, calibrated, oof_labels)
    default_policy = DecisionPolicy(final_model.alpha)
    default_policy.held = policy.held
    default_takes = default_policy.predict(oof_features, calibrated)
    takes = policy.predict(oof_features, calibrated)
    gains = pnl(takes, oof_labels.max_x.to_numpy(), oof_labels.final_x.to_numpy())
    held5 = policy.held.predict(calibrated)[:, 2]
    p5 = reach_probability(calibrated, 5., final_model.alpha)
    baseline = np.where(p5 > -held5/(3.7-held5), 5., 0.)
    report = pd.DataFrame({'day': day_names, 'n_tokens': 1, 'n_buys': takes>0,
        'n_hits': (takes>0) & (oof_labels.max_x.to_numpy() >= takes), 'pnl': gains,
        'baseline_pnl': pnl(baseline, oof_labels.max_x.to_numpy(), oof_labels.final_x.to_numpy()),
        'default_pnl': pnl(default_takes, oof_labels.max_x.to_numpy(), oof_labels.final_x.to_numpy())})
    table = report.groupby('day', sort=True).sum().reset_index()
    table['cum_pnl'] = table.pnl.cumsum()
    table['tuning'] = 'in-sample-tuned on training OOF'
    return FittedArtifacts(final_model, calibration, policy, training_features.columns.tolist(),
                           train, universe, creation, early, base, table)


def run_pipeline(data_dir=None, out_paths=None, wf_days=7, archive_days=30, use_lightgbm=True,
                 return_artifacts=False):
    """Return (submission, walk-forward table), with zero-take safety fallback.

    Fold histories see only that fold's eligible labels. Calibration and policy
    tune on pooled temporal OOF, so reported policy PnL is in-sample-tuned.
    No synthetic answer file is ever read here.
    With return_artifacts=True, append the in-memory fit (or None on no fit).
    """
    if out_paths is None:
        out_paths = ['submission.csv', '/kaggle/working/submission.csv']
    table = pd.DataFrame(columns=['day', 'n_tokens', 'n_buys', 'n_hits', 'pnl', 'cum_pnl', 'default_pnl', 'baseline_pnl'])
    submission = pd.DataFrame({'mint': pd.Series(dtype=str), 'tp': pd.Series(dtype=float)})
    artifacts = None
    try:
        try:
            root = Path(data_dir) if data_dir is not None else discover_data_dir()
        except FileNotFoundError:
            root = Path(data_dir) if data_dir else Path('data')
        test = _read(root, 'test.csv', ['mint', 'mt'])
        if not (root/'test.csv').exists():
            test = _read(root, 'sample_submission.csv', ['mint']).reindex(columns=['mint', 'mt'])
        submission = test[['mint']].assign(tp=0.)
        train = load_training_labels(root, archive_days)
        if len(train) < 200 or test.empty or test.mt.isna().any():
            print(f'Zero-take fallback: {len(train)} labeled rows or missing test times')
        else:
            artifacts = fit_pipeline(root, wf_days, archive_days, use_lightgbm, train, test)
            table = artifacts.walk_forward
            history = history_features(test, train, artifacts.universe, artifacts.early_buyers,
                                       artifacts.creation).set_index('mint')
            test_features = artifacts.base_features.reindex(test.mint).reset_index().merge(
                history.reset_index(), on='mint', how='left')
            submission['tp'] = artifacts.predict_features(test_features)
            validate_submission(submission, test)
            print(f'Backend: {artifacts.model.backend}; margin={artifacts.policy.margin:g}; exclusions={artifacts.policy.rules}')
    except Exception as error:
        print(f'Zero-take fallback: {type(error).__name__}: {error}')
        submission['tp'] = 0.
    _write_submission(submission, out_paths)
    return (submission, table, artifacts) if return_artifacts else (submission, table)


In [ ]:
submission, walk_forward = run_pipeline()
print('Walk-forward: pnl is in-sample-tuned on training OOF; default_pnl has margin=0 and no exclusions.')
print(walk_forward.to_string(index=False))
if len(walk_forward):
    print('PnL totals:', walk_forward[['pnl', 'default_pnl', 'baseline_pnl']].sum().to_dict())
print(f'Submission: {len(submission)} tokens; n buys={int(submission.tp.gt(0).sum())}')
print('tp distribution:')
print(submission.tp.value_counts().sort_index().to_string())
print('Wrote ./submission.csv; attempted /kaggle/working/submission.csv.')
